# WeaponDetect — YOLOv8 Training on Colab

Trains a custom weapon-detection YOLOv8 model for the **weapondetect** app.

**Dataset:** *Yolo Weapon Detection* v8 by `weapon-detect-qbsiw` (Roboflow Universe) — 10,833 images @640px, 9 raw classes, **CC BY 4.0** (attribution required — credit the dataset in the app's README/About page).

**Before running:**
1. In Colab: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**.
2. Create a **free** Roboflow account at https://app.roboflow.com and copy your API key (Settings → API). You will paste it when prompted — it is never saved anywhere.
3. Expected runtime: ~1–2 hours for 50 epochs of `yolov8n` on a T4. `yolov8s` is slower but more accurate — your choice in the training cell.

**Outputs:** `best.pt` (custom checkpoint) + `metrics.json` (real evaluation for the app's Analytics page).

In [ ]:
# 1 — Verify GPU is available
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — set Runtime > Change runtime type > T4 GPU, then re-run.")

In [ ]:
# 2 — Install dependencies
!pip install -q roboflow ultralytics pyyaml

In [ ]:
# 3 — Download the dataset (YOLOv8 format)
from getpass import getpass
from roboflow import Roboflow

api_key = getpass("Paste your free Roboflow API key (app.roboflow.com → Settings → API): ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("weapon-detect-qbsiw").project("yolo-weapon-detection")
version = project.version(8)
dataset = version.download("yolov8")
print("Downloaded to:", dataset.location)

## 4 — Validate the dataset
Checks image↔label correspondence, class-id range, and that every box is normalized to [0, 1].

In [ ]:
from pathlib import Path
import yaml

loc = Path(dataset.location)
with open(loc / "data.yaml") as f:
    data = yaml.safe_load(f)
print("classes:", data["names"])
print("nc:", data["nc"])

def check_split(split):
    img_dir, lbl_dir = loc / split / "images", loc / split / "labels"
    imgs = list(img_dir.glob("*")) if img_dir.exists() else []
    missing, bad = 0, 0
    for img in imgs:
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            missing += 1
            continue
        for line in lbl.read_text().splitlines():
            p = line.split()
            if len(p) != 5:
                bad += 1; continue
            cid, *box = p
            if int(float(cid)) >= data["nc"]:
                bad += 1; continue
            if any(not (0.0 <= float(v) <= 1.0) for v in box):
                bad += 1
    print(f"{split}: {len(imgs)} images | {missing} missing label files | {bad} malformed lines")

for s in ("train", "valid", "test"):
    check_split(s)

## 5 — Clean the class list
`blunt object` has only ~3 annotations — unlearnable, so we drop it (by name, robust to id order) and remap the remaining class ids. The other 8 classes are kept as labeled; note that `submachine-gun` is rare (~46 annotations), so expect weaker scores there — documented in the final report, not hidden.

In [ ]:
import shutil

DROP = {"blunt object"}
names = data["names"]
keep = [(i, n) for i, n in enumerate(names) if n not in DROP]
remap = {old: new for new, (old, _) in enumerate(keep)}
new_names = [n for _, n in keep]
print("keeping", len(new_names), "classes:", new_names)

clean = loc.parent / (loc.name + "-clean")
if clean.exists():
    shutil.rmtree(clean)
for split in ("train", "valid", "test"):
    for kind in ("images", "labels"):
        src = loc / split / kind
        if src.exists():
            shutil.copytree(src, clean / split / kind)
    for lf in (clean / split / "labels").glob("*.txt"):
        out = []
        for line in lf.read_text().splitlines():
            p = line.split()
            if len(p) != 5:
                continue
            old = int(float(p[0]))
            if old not in remap:
                continue  # dropped class
            out.append(str(remap[old]) + " " + " ".join(p[1:]))
        lf.write_text("\n".join(out) + ("\n" if out else ""))

clean_yaml = clean / "data.yaml"
with open(clean_yaml, "w") as f:
    yaml.safe_dump({
        "path": str(clean),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(new_names),
        "names": new_names,
    }, f)
print("clean data.yaml →", clean_yaml)

## 6 — Train
`yolov8n` (fast, Colab-friendly) is the default. Swap to `"yolov8s.pt"` for higher accuracy if you have time. If you hit CUDA out-of-memory, lower `batch` to 8.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # COCO-pretrained starting point
results = model.train(
    data=str(clean_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/runs",
    name="weapondetect",
    patience=15,
    seed=42,
)
best = str(results.save_dir / "weights" / "best.pt")
print("best checkpoint →", best)

## 6b — Guard: mirror checkpoints to Google Drive

Free Colab runtimes die when idle (~90 min) and have hard session limits; everything under `/content` — including `runs/weapondetect/weights/best.pt` and `last.pt` — vanishes with the runtime. This cell mirrors the run directory to Drive. Run it right after training finishes (and re-run it any time for a fresh snapshot). If the runtime dies mid-training, re-mount Drive, copy the mirror back, and resume from `last.pt` using the commented recipe below.


In [ ]:
# Guard — mirror checkpoints to Google Drive so a disconnect can't wipe the run
from google.colab import drive
drive.mount("/content/drive")  # one-time sign-in per session

from pathlib import Path
import shutil

RUN_DIR = Path("/content/runs/weapondetect")
MIRROR = Path("/content/drive/MyDrive/weapondetect/runs/weapondetect")
if Path("/content/drive/MyDrive").exists():
    MIRROR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUN_DIR, MIRROR, dirs_exist_ok=True)
    print("Mirrored", RUN_DIR, "→", MIRROR)
else:
    print("Drive not mounted — run drive.mount(\"/content/drive\") first, then re-run this cell.")
print("Resume recipe if the runtime dies mid-training:")
print("  shutil.copytree(MIRROR, RUN_DIR, dirs_exist_ok=True)")
print('  model = YOLO(str(RUN_DIR / "weights" / "last.pt"))')
print("  model.train(resume=True)")



## 7 — Evaluate on the held-out test split
Produces **real** metrics (precision, recall, F1, mAP@0.50, mAP@0.50:0.95, per-class AP, latency) and writes `metrics.json` in the exact format the app's **Analytics** page and `GET /api/metrics` consume. Nothing here is invented — every number comes from this run.

In [ ]:
import json, time
from datetime import datetime, timezone

model = YOLO(best)
val = model.val(data=str(clean_yaml), split="test", device=0)

mp, mr = float(val.box.mp), float(val.box.mr)
map50, map5095 = float(val.box.map50), float(val.box.map)
f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0

per_class = []
for i, n in enumerate(new_names):
    per_class.append({
        "class": n,
        "precision": round(float(val.box.p[i]), 4),
        "recall": round(float(val.box.r[i]), 4),
        "ap50": round(float(val.box.ap50[i]), 4),
        "ap50_95": round(float(val.box.ap[i]), 4),
    })

# inference latency: mean over up to 20 test images
test_imgs = sorted((clean / "test" / "images").glob("*"))[:20]
t0 = time.perf_counter()
model.predict([str(p) for p in test_imgs], device=0, verbose=False)
lat_ms = (time.perf_counter() - t0) / max(len(test_imgs), 1) * 1000

metrics = {
    "available": True,
    "model": "yolov8n-weapondetect",
    "model_family": "YOLOv8",
    "checkpoint": "best.pt",
    "dataset": "weapon-detect-qbsiw/yolo-weapon-detection v8 (cleaned: dropped 'blunt object')",
    "dataset_license": "CC BY 4.0 — attribution: 'Yolo Weapon Detection' by weapon-detect-qbsiw, Roboflow Universe",
    "classes": new_names,
    "precision": round(mp, 4),
    "recall": round(mr, 4),
    "f1": round(f1, 4),
    "map50": round(map50, 4),
    "map50_95": round(map5095, 4),
    "per_class": per_class,
    "inference_latency_ms": round(lat_ms, 1),
    "device": "cuda",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "notes": "Test-split evaluation. 'submachine-gun' is rare (~46 annotations) — treat its scores cautiously.",
}
with open("/content/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps({k: metrics[k] for k in ("precision", "recall", "f1", "map50", "map50_95", "inference_latency_ms")}, indent=2))

In [ ]:
# 8 — Download the artifacts
from google.colab import files
from pathlib import Path
import shutil
files.download(best)
files.download("/content/metrics.json")
print("Downloaded best.pt and metrics.json — see below for wiring them into the app.")
# Also keep final artifacts on Drive (files.download only works while the runtime is alive)
if Path("/content/drive/MyDrive").exists():
    final = Path("/content/drive/MyDrive/weapondetect/artifacts")
    final.mkdir(parents=True, exist_ok=True)
    shutil.copy(best, final / "best.pt")
    shutil.copy("/content/metrics.json", final / "metrics.json")
    print("Final artifacts also on Drive:", final)
else:
    print("Drive not mounted — artifacts downloaded locally only.")




## 9 — Wire into the weapondetect app

1. Put `best.pt` in the app's `models/` folder (it's gitignored — never commit weights).
2. Set `MODEL_PATH` to it, e.g. in `backend/.env`: `MODEL_PATH=/absolute/path/to/models/best.pt`
3. Copy `metrics.json` to `training/evaluation/reports/metrics.json` in the repo.
4. Restart the backend.

Result: `GET /api/model-info` reports your real weapon classes, image detection uses your custom model, and the Analytics dashboard renders your real evaluation numbers.

**Attribution (required by CC BY 4.0):** credit *“Yolo Weapon Detection” by weapon-detect-qbsiw, Roboflow Universe* in the README and About page.